In [0]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

expense_rows = [{
    "expense_type": "maintenance",
    "expense_basis": "premium",
    "expense_rate": 0.03,   # 3%
    "version_id": "EXPENSE_2026_04",
    "comment": "3% maintenance expense on premium"
}]

df_expense = pd.DataFrame(expense_rows)

spark.createDataFrame(df_expense) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.life_insurance_raw.expense_assumption")


In [0]:
display(df_expense)

In [0]:
stress_rows = [
    # Core Scenarios
    ("BASE", "Base", 1.00, 1.00, 0, "Core", True, "Base scenario"),
    ("MORT_UP", "Mortality", 1.10, 1.00, 0, "Core", False, "Mortality +10%"),
    ("MORT_DOWN", "Mortality", 0.90, 1.00, 0, "Core", False, "Mortality -10%"),
    ("LAPSE_UP", "Lapse", 1.00, 1.10, 0, "Core", False, "Lapse +10%"),
    ("LAPSE_DOWN", "Lapse", 1.00, 0.90, 0, "Core", False, "Lapse -10%"),
    ("RATE_UP", "Discount", 1.00, 1.00, 50, "Core", False, "Discount +50bps"),
    ("RATE_DOWN", "Discount", 1.00, 1.00, -50, "Core", False, "Discount -50bps"),
    # Advanced Scenarios
    ("COMBINED_ADVERSE", "Combined", 1.10, 1.00, -50, "Advanced", False, "Mortality↑, Discount↓"),
    ("PERSIST_IMPROVE", "Lapse", 1.00, 0.70, 0, "Advanced", False, "Persistency improves (lapse -30%)"),
]

df_stress = pd.DataFrame(stress_rows, columns=[
    "scenario_id",
    "scenario_type",
    "mortality_multiplier",
    "lapse_multiplier",
    "discount_shift_bps",
    "scenario_group",
    "is_base_scenario",
    "scenario_description"
])

spark.createDataFrame(df_stress) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.life_insurance_raw.stress_scenario_definition")


In [0]:
display(df_stress)

In [0]:
version_rows = [
    ("mortality", "MORT_2026_04", "Mortality assumption v2026.04", True),
    ("lapse", "LAPSE_2026_04", "Lapse assumption v2026.04", True),
    ("discount", "RFR_20251231", "EIOPA RFR 2025-12-31", True),
    ("expense", "EXPENSE_2026_04", "Expense assumption v2026.04", True),
    ("pricing", "PRICING_2026_04", "Pricing assumption v2026.04", True),
]

df_version = pd.DataFrame(version_rows, columns=[
    "assumption_type",
    "version_id",
    "version_name",
    "is_base_version"
])

spark.createDataFrame(df_version) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.life_insurance_raw.assumption_version")


In [0]:
display(df_version)